# Lesson 7.3 — The Twin-in-the-Loop Cycle

One full turn: **monitor -> re-sync if drifted -> predict -> adapt**. Composition of verified operators; no new theory.

In [1]:
import numpy as np
from modules.module09.integration import GreenhouseWorld, harvest_row
from modules.module10.twin import (GroundTruth, DigitalTwin, snapshot,
    prevalidate, select_action, twin_in_the_loop)
mk = lambda: GreenhouseWorld.demo_row(n=5, seed=1)
F2 = mk().fruit[2]
OBST = {'F2': {'obstacle': ((float(F2.x), float(F2.y)), 0.05)}}
def effort(rep): return sum(p['n_attempts'] for p in rep['picks'])
checks = []
twin = DigitalTwin(mk()); state = snapshot(mk()); twin.sync(state)
candidates = {'clear_path': None, 'ignore_obstacle': OBST}
# In sync: monitor quiet, no re-sync, still returns a chosen action
turn = twin_in_the_loop(twin, state, candidates)
print('in-sync: alert =', turn['monitor']['alert'], '| resynced =', turn['resynced'],
      '| chosen =', turn['chosen'])
checks.append(turn['monitor']['alert'] is False)
checks.append(turn['resynced'] is False)
checks.append(turn['chosen'] == 'clear_path')


in-sync: alert = False | resynced = False | chosen = clear_path


In [2]:
# Reality drifts: monitor alerts, the loop re-syncs, then decides from the corrected state
real = GroundTruth(mk()); real.world.q = np.asarray(real.world.q, float) + np.array([0.08, -0.04])
drift = snapshot(real.world)
turn2 = twin_in_the_loop(twin, drift, candidates)
print('drifted: alert =', turn2['monitor']['alert'], '| resynced =', turn2['resynced'],
      '| chosen =', turn2['chosen'])
checks.append(turn2['monitor']['alert'] is True)
checks.append(turn2['resynced'] is True)
checks.append(turn2['chosen'] is not None)


drifted: alert = True | resynced = True | chosen = clear_path


In [3]:
assert all(checks), checks
print('All checks passed.')


All checks passed.
